## **BOILER PLATE**

In [22]:
import os
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv(dotenv_path="../.env",override=True)

api_key=os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY is missing from .env")

# llm = ChatOpenRouter(
#     model="deepseek/deepseek-v4-flash-0731",
#     api_key=api_key,
#     temperature=1.2,
#     max_tokens=500,
# )

In [23]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_review_flag: Literal["positive","negative"]


## **CHAIN WITH CONDITIONAL PROCESSINGS**

In [24]:
#TASK-1 prompt

from langchain_core.prompts import ChatPromptTemplate

prompt_template_1 = ChatPromptTemplate([
    ("system","You are a movie review Evaluator"),
    ("human","Please catagorize this movie review as 'positive' or 'negative' :{input}")
])

In [25]:
#TASK-2 llm

llm_1 = ChatOpenRouter(
    model="deepseek/deepseek-v4-flash-0731",
    api_key=api_key,
    temperature=0,
    max_tokens=2000,
)

llm_1_structured = llm_1.with_structured_output(llm_schema)

# llm_1_structured.invoke("This movie was good")

In [26]:
#TASK-3  Custom Runnable
from langchain_core.runnables import RunnableLambda

def pydantic_json(input:llm_schema)-> str:
    return input.model_dump()['movie_review_flag']

pydantic_json_runnable = RunnableLambda(pydantic_json)


### **CONDITIONAL CHAIN - 1**

In [27]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence,RunnableLambda

#TASK-1 prompt

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a LinkedIn Social Media Manager"),
    ("human","Create a post for the following text: {text}")
])

#TASK-2 llm

llm_2 = ChatOpenRouter(
    model="deepseek/deepseek-v4-flash-0731",
    api_key=api_key,
    temperature=1.0,
    max_tokens=20000,
)

#TASK-3 Parser
str_parser_2 = StrOutputParser()

#TASK-4 chain

chain_linkedin = RunnableSequence(linkedin_prompt,llm_2,str_parser_2)

### **CONDITIONAL CHAIN 2**

In [31]:
def instagram_chain(text:str):

    #TASK-1 prompt

    instagram_prompt = ChatPromptTemplate.from_messages([
        ("system","You are a Instagram Social Media Manager"),
        ("human","Create a post for the following text: {text}")
    ])

    #TASK-2 llm

    llm_3 = ChatOpenRouter(
        model="deepseek/deepseek-v4-flash-0731",
        api_key=api_key,
        temperature=1.2,
        max_tokens=20000,
    )

    #TASK-3 parser

    str_parser_3 = StrOutputParser()

    #TASK-4 chain

    chain_instagram = RunnableSequence(instagram_prompt,llm_3,str_parser_3)

    result = chain_instagram.invoke(text)

    return result

instagram_chain_runnable = RunnableLambda(instagram_chain)

## **FINAL CHAIN**

In [37]:
from langchain_core.load.dump import default
from langchain_core.runnables import RunnableBranch

conditional_chain = RunnableBranch(
    (lambda x:"positive" in x, chain_linkedin),
    instagram_chain_runnable
)


final_orchestrator = RunnableSequence(prompt_template_1,llm_1_structured,pydantic_json_runnable,conditional_chain)

final_orchestrator.invoke({"input":"KGF Good Movie"})

'Here’s a LinkedIn post built around the theme of positivity, designed to be engaging and shareable:\n\n---\n\nPositivity is more than just a mindset—it’s a strategy.\n\nI’ve seen it transform teams, spark creativity, and turn challenges into opportunities. It’s not about ignoring the hard stuff. It’s about choosing to focus on what *can* be done instead of what can’t.\n\nWhen we bring positive energy to work, we don’t just feel better—we perform better. We collaborate more openly, solve problems more effectively, and lead with more purpose.\n\nSo here’s my challenge to you today: find one moment to be the positive force in someone’s professional life. A kind word, a helpful hand, a bit of genuine encouragement.\n\nYou might be surprised by how far it travels.\n\nWhat does positivity look like in your workplace? Let’s talk in the comments. 👇\n\n#Positivity #Mindset #Leadership #WorkplaceCulture #GrowthMindset'